# Beyond Kubernetes Jobs: live demo

Step-by-step companion to the slides. Every **▶ NB x.y** chip on a slide is a section here, in the same order.

**Before the talk**
1. Install `docker`, `kind`, `kubectl`, `jq`, and Python 3 with Jupyter (`pip install jupyterlab`).
2. Start Jupyter **from the `demo/` directory**: `cd demo && jupyter lab jobset-demo.ipynb`
3. Run **NB 0** once. Creating the cluster can take a few minutes the first time.

| NB | Slide | What you show |
|---|---|---|
| 0 | 9 | The demo cluster: WAS on, 3 workers × 2 fake GPUs |
| 1.1 | 4 | A plain Job |
| 2.1 – 2.5 | 10 – 15 | `hello`: roles, tree, defaults, events, DNS, coordinator |
| 3.1 | 17 | Success policy |
| 3.2 – 3.5 | 18 – 19 | `resilient`: three failures, three behaviours |
| 4.1 | 22 | `pipeline`: dependsOn |
| 5.1 – 5.2 | 27 – 28 | `train`: Workload, PodGroups, two trees |
| 5.3 – 5.4 | 29 | Partial placement vs. gang |
| 5.5 | 30 | Elastic scale-down, `minCount` follows |
| 5.6 | 31 | Suspend / resume |
| 6 | – | Clean up |

## ▶ NB 0 · Slide 9: One cluster for every demo

Helpers first. `sh()` prints each command before running it, so the audience sees exactly what you typed.

In [ ]:
"""Helpers for the live demo. Run this cell first."""
import json
import pathlib
import subprocess
import time

from IPython.display import Markdown, display

DEMO_DIR = pathlib.Path.cwd()
MANIFESTS = DEMO_DIR / "manifests"
CONTEXT = "kind-jobset-demo"
JOBSET = "jobset.sigs.k8s.io/jobset-name"
RJOB = "jobset.sigs.k8s.io/replicatedjob-name"
JOB_INDEX = "jobset.sigs.k8s.io/job-index"
KIND_ORDER = {"Workload": 0, "PodGroup": 1, "Service": 2, "Job": 3, "Pod": 4}

assert (MANIFESTS / "01-hello.yaml").exists(), "Start Jupyter from the demo/ directory"


def sh(cmd, timeout=900):
    """Print a shell command, run it, and stream its output."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=DEMO_DIR, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"[timed out after {timeout}s]")
    if proc.returncode:
        print(f"[exit code {proc.returncode}]")
    print()
    return "".join(lines)


def show(manifest):
    """Render a manifest from demo/manifests as highlighted YAML."""
    text = (MANIFESTS / manifest).read_text()
    display(Markdown(f"**`manifests/{manifest}`**\n\n```yaml\n{text}\n```"))


def kjson(args):
    """`kubectl get <args> -o json`, parsed. Returns an empty list on error."""
    result = subprocess.run(f"kubectl get {args} -o json", shell=True, text=True, capture_output=True)
    if result.returncode:
        return {"items": []}
    return json.loads(result.stdout)


def live(objs):
    """Drop objects that are being deleted."""
    return [o for o in objs if not o["metadata"].get("deletionTimestamp")]


def pods(jobset, rjob=None):
    selector = f"{JOBSET}={jobset}" + (f",{RJOB}={rjob}" if rjob else "")
    return live(kjson(f"pods -l {selector}")["items"])


def running(jobset, rjob=None):
    return [p for p in pods(jobset, rjob) if p["status"].get("phase") == "Running"]


def running_names(jobset):
    return {p["metadata"]["name"] for p in running(jobset)}


def pod_name(jobset, rjob, job_index=0):
    """Name of the live pod belonging to Job <job_index> of a ReplicatedJob."""
    matches = [p for p in pods(jobset, rjob) if p["metadata"]["labels"].get(JOB_INDEX) == str(job_index)]
    if not matches:
        raise RuntimeError(f"no live pod for {jobset}/{rjob}/{job_index}")
    return matches[0]["metadata"]["name"]


def wait_until(what, check, timeout=180, every=2):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if check():
            print(f"✓ {what}")
            return True
        time.sleep(every)
    print(f"✗ timed out waiting for: {what}")
    return False


def wait_running(jobset, count, rjob=None, timeout=180):
    scope = f"{jobset}/{rjob}" if rjob else jobset
    return wait_until(f"{count} pods running in {scope}", lambda: len(running(jobset, rjob)) == count, timeout)


def wait_replaced(jobset, before, count, timeout=180):
    """Wait until `count` pods run again and the set of pods has changed."""
    def replaced():
        now = running_names(jobset)
        return len(now) == count and now != before
    return wait_until(f"{jobset} pods replaced and running again", replaced, timeout)


def owner_tree(jobset):
    """Print everything a JobSet owns by following ownerReferences (like `kubectl tree`)."""
    root = kjson(f"jobset {jobset}")
    if "metadata" not in root:
        print(f"JobSet {jobset} not found")
        return
    kinds = "services,jobs,pods,workloads.scheduling.k8s.io,podgroups.scheduling.k8s.io"
    children = {}
    for obj in live(kjson(kinds)["items"]):
        for ref in obj["metadata"].get("ownerReferences", []):
            children.setdefault(ref["uid"], []).append(obj)

    def label(obj):
        name = f'{obj["kind"]}/{obj["metadata"]["name"]}'
        return f'{name}  [{obj["status"].get("phase")}]' if obj["kind"] == "Pod" else name

    def walk(obj, prefix):
        kids = sorted(children.get(obj["metadata"]["uid"], []),
                      key=lambda o: (KIND_ORDER.get(o["kind"], 9), o["metadata"]["name"]))
        for i, kid in enumerate(kids):
            last = i == len(kids) - 1
            print(prefix + ("└─ " if last else "├─ ") + label(kid))
            walk(kid, prefix + ("   " if last else "│  "))

    print(label(root))
    walk(root, "")


def scheduling_tree(jobset):
    """Print Workload -> PodGroupTemplates -> PodGroups -> Pods for a JobSet."""
    selector = f"-l {JOBSET}={jobset}"
    groups = live(kjson(f"podgroups.scheduling.k8s.io {selector}")["items"])
    members = pods(jobset)
    workloads = live(kjson(f"workloads.scheduling.k8s.io {selector}")["items"])
    if not workloads:
        print(f"no Workload for {jobset} (suspended, finished, or no spec.scheduling)")
    for workload in workloads:
        print(f'Workload/{workload["metadata"]["name"]}')
        templates = workload["spec"]["podGroupTemplates"]
        for ti, tmpl in enumerate(templates):
            pad = "   " if ti == len(templates) - 1 else "│  "
            gang = tmpl["schedulingPolicy"].get("gang")
            policy = f'gang minCount={gang["minCount"]}' if gang else "basic"
            print(("└─ " if pad == "   " else "├─ ") + f'template {tmpl["name"]}  ({policy})')
            for group in groups:
                ref = group["spec"].get("workloadRef", {}).get("templateName")
                if ref != tmpl["name"] and not group["metadata"]["name"].endswith(tmpl["name"]):
                    continue
                conditions = {c["type"]: c for c in group.get("status", {}).get("conditions", [])}
                scheduled = conditions.get("PodGroupInitiallyScheduled", {})
                state = f'{scheduled.get("status", "?")} {scheduled.get("reason", "")}'.strip()
                group_name = group["metadata"]["name"]
                print(pad + f"└─ PodGroup/{group_name}  [scheduled: {state}]")
                in_group = sorted((p for p in members
                                   if (p["spec"].get("schedulingGroup") or {}).get("podGroupName") == group_name),
                                  key=lambda p: p["metadata"]["name"])
                for pi, pod in enumerate(in_group):
                    branch = "└─ " if pi == len(in_group) - 1 else "├─ "
                    job = pod["metadata"]["labels"].get("batch.kubernetes.io/job-name", "?")
                    print(pad + "   " + branch + f'Pod/{pod["metadata"]["name"]}  [{pod["status"].get("phase")}]  job={job}')


context = subprocess.run("kubectl config current-context", shell=True, text=True, capture_output=True).stdout.strip()
print(f"kubectl context: {context or '(none)'}")
if context and context != CONTEXT:
    print(f"⚠️  expected {CONTEXT}: run NB 0 to create it, or `kubectl config use-context {CONTEXT}`")

Create (or reuse) the cluster: kind with Kubernetes 1.37 and Workload-Aware Scheduling, 3 workers with 2 fake GPUs each, and JobSet with its alpha gates on. Safe to re-run.

In [ ]:
sh("./cluster.sh up", timeout=1800)

Show the slide's status view, plus the JobSet controller config as YAML:

In [ ]:
sh("./cluster.sh status")
sh("kubectl -n jobset-system get configmap jobset-manager-config -o yaml")

## ▶ NB 1.1 · Slide 4: A Job runs pods until they finish

In [ ]:
show("00-plain-job.yaml")
sh("kubectl apply -f manifests/00-plain-job.yaml")
sh("kubectl wait --for=condition=Complete job/hello-batch --timeout=120s")
sh("kubectl get pods -l job-name=hello-batch")
sh("kubectl logs -l job-name=hello-batch --prefix")

In [ ]:
sh("kubectl delete -f manifests/00-plain-job.yaml --wait")

## ▶ NB 2.1 · Slide 10: One manifest, two roles, DNS, coordinator, success policy

Ask the room to predict: how many Jobs, Pods and Services?

In [ ]:
show("01-hello.yaml")
sh("kubectl apply -f manifests/01-hello.yaml")

## ▶ NB 2.2 · Slide 12: Apply it: 3 Jobs, 5 Pods

The workers run for about 90 seconds, which leaves time for 2.3 – 2.5 before the success policy kicks in.

In [ ]:
wait_running("hello", 5)
sh("kubectl get jobset hello")
sh(f"kubectl get jobs -l {JOBSET}=hello")
sh(f"kubectl get pods -l {JOBSET}=hello -L {RJOB},{JOB_INDEX}")

## ▶ NB 2.3 · Slide 13: What actually exists in the cluster

The ownership tree, built from `ownerReferences`. There's no ReplicatedJob object, only Jobs labelled with it.

In [ ]:
owner_tree("hello")
sh("kubectl tree jobset hello 2>/dev/null || echo '(optional: kubectl krew install tree)'")

## ▶ NB 2.4 · Slide 14: Watch the background work happen

Point at the fields you never wrote (`startupPolicy`, `completionMode`, `restartPolicy`, `network`): the webhook added them.

In [ ]:
sh("kubectl get jobset hello -o yaml")

In [ ]:
sh("kubectl get events --sort-by=.lastTimestamp "
   "-o custom-columns=FROM:.reportingComponent,REASON:.reason,OBJECT:.involvedObject.name | tail -15")
sh("kubectl get jobset hello -o json | jq -c '.status.replicatedJobsStatus[] | {name, ready, active, succeeded}'")

## ▶ NB 2.5 · Slide 15: Stable names, and a coordinator everyone can find

In [ ]:
sh("kubectl get svc hello")
sh(f"kubectl get pods -l {JOBSET}=hello -o custom-columns=HOSTNAME:.spec.hostname,SUBDOMAIN:.spec.subdomain")
sh("kubectl run dns-test --rm -i --restart=Never --image=busybox:1.37 -- nslookup hello-workers-1-0.hello")

In [ ]:
sh(f"kubectl get pods -l {JOBSET}=hello -L jobset.sigs.k8s.io/coordinator")
sh(f"kubectl logs -l {RJOB}=workers,{JOBSET}=hello --prefix")

## ▶ NB 3.1 · Slide 17: Done when the real work is done

The leader would sleep for an hour. The success policy completes the JobSet when `workers` finish and tears the leader down.

In [ ]:
sh("kubectl wait --for=condition=Completed jobset/hello --timeout=240s")
sh("kubectl get jobset hello")
sh(f"kubectl get jobs -l {JOBSET}=hello")
sh("kubectl get jobset hello -o jsonpath='{range .status.conditions[*]}{.type}={.status} {.reason}{\"\\n\"}{end}'")

In [ ]:
sh("kubectl delete jobset hello --wait")

## ▶ NB 3.2 · Slides 18 – 19: One failure policy, three behaviours

Every pod waits for a trigger file: `/tmp/crash` exits 1 (a bug), `/tmp/preempt` exits 42 (simulated preemption, matched by the Job's `podFailurePolicy`).

In [ ]:
show("02-resilient.yaml")
sh("kubectl apply -f manifests/02-resilient.yaml")
wait_running("resilient", 4)
sh(f"kubectl get pods -l {JOBSET}=resilient")

## ▶ NB 3.3 · ① A worker crashes → `RestartJob`

Only `workers-1` is recreated. Check the AGE column.

In [ ]:
before = running_names("resilient")
sh(f"kubectl exec {pod_name('resilient', 'workers', 1)} -- touch /tmp/crash")
wait_replaced("resilient", before, 4)
sh(f"kubectl get pods -l {JOBSET}=resilient")
sh("kubectl get jobset resilient -o json | jq -c '.status | {restarts, restartsCountTowardsMax, "
   "workers: [.replicatedJobsStatus[] | select(.name==\"workers\") | .jobRestarts]}'")

## ▶ NB 3.4 · ② The leader is "preempted" → free `RestartJobSet`

Every pod is new, `restarts` goes up, and `restartsCountTowardsMax` does not.

In [ ]:
before = running_names("resilient")
sh(f"kubectl exec {pod_name('resilient', 'leader')} -- touch /tmp/preempt")
wait_replaced("resilient", before, 4)
sh(f"kubectl get pods -l {JOBSET}=resilient")
sh("kubectl get jobset resilient -o json | jq -c '.status | {restarts, restartsCountTowardsMax}'")

## ▶ NB 3.5 · ③ The leader has a bug → `RestartJobSet`, spends the budget

In [ ]:
before = running_names("resilient")
sh(f"kubectl exec {pod_name('resilient', 'leader')} -- touch /tmp/crash")
wait_replaced("resilient", before, 4)
sh(f"kubectl get pods -l {JOBSET}=resilient")
sh("kubectl get jobset resilient -o json | jq -c '.status | {restarts, restartsCountTowardsMax}'")
sh("kubectl describe jobset resilient | sed -n '/Events:/,$p'")

*Optional:* keep crashing the leader until the budget (`maxRestarts: 3`) is gone and the JobSet fails with `ReachedMaxRestarts`.

In [ ]:
for _ in range(3):
    if not running("resilient", "leader"):
        break
    before = running_names("resilient")
    sh(f"kubectl exec {pod_name('resilient', 'leader')} -- touch /tmp/crash")
    wait_until("leader restarted or JobSet failed",
               lambda: running_names("resilient") != before or not running("resilient"), timeout=120)
sh("kubectl wait --for=condition=Failed jobset/resilient --timeout=120s")
sh("kubectl get jobset resilient")
sh("kubectl get jobset resilient -o jsonpath='{range .status.conditions[*]}{.type}={.status} {.reason}{\"\\n\"}{end}'")

In [ ]:
sh("kubectl delete jobset resilient --wait")

## ▶ NB 4.1 · Slide 22: Watch the dependency graph unfold

`initializer` must **Complete** → `launcher` must be **Ready** → then the `worker` Job is created.

In [ ]:
show("03-pipeline.yaml")
sh("kubectl apply -f manifests/03-pipeline.yaml")
for _ in range(6):
    sh(f"kubectl get jobs -l {JOBSET}=pipeline")
    if any(p["metadata"]["labels"].get(RJOB) == "worker" for p in pods("pipeline")):
        break
    time.sleep(8)

In [ ]:
sh("kubectl delete jobset pipeline --wait")

## ▶ NB 5.1 · Slide 27: Turn it on: one gang per role

`train` holds 4 of the 6 GPUs and keeps running for the rest of section 5.

In [ ]:
show("04-train.yaml")
sh("kubectl apply -f manifests/04-train.yaml")
wait_running("train", 5)
sh(f"kubectl get workloads.scheduling.k8s.io,podgroups -l {JOBSET}=train")
sh(f"kubectl get podgroups -l {JOBSET}=train -o custom-columns="
   "NAME:.metadata.name,TEMPLATE:.spec.workloadRef.templateName,MIN:.spec.schedulingPolicy.gang.minCount")

The generated objects as YAML: the Workload (blueprint) and the PodGroups (runtime instances).

In [ ]:
sh(f"kubectl get workloads.scheduling.k8s.io -l {JOBSET}=train -o yaml")
sh(f"kubectl get podgroups -l {JOBSET}=train -o yaml")

How the pieces are wired: the Job annotation and each pod's `schedulingGroup`.

In [ ]:
sh("kubectl get job train-workers-1 -o jsonpath="
   "'{.metadata.annotations.scheduling\\.k8s\\.io/group-template-name}{\"\\n\"}'")
sh(f"kubectl get pods -l {JOBSET}=train -o custom-columns="
   "POD:.metadata.name,PODGROUP:.spec.schedulingGroup.podGroupName,NODE:.spec.nodeName")

## ▶ NB 5.2 · Slide 28: Who owns it vs. who gets scheduled together

Same pods, two trees. Note the `workers` gang spans **two Jobs**.

In [ ]:
owner_tree("train")
print()
scheduling_tree("train")

## ▶ NB 5.3 · Slide 29 (left): Without gang: partial placement

Only 2 GPUs are free. The same workload without `spec.scheduling` grabs them anyway.

In [ ]:
show("05-big-no-gang.yaml")
sh("kubectl apply -f manifests/05-big-no-gang.yaml")
wait_running("big-no-gang", 2, rjob="workers")
time.sleep(5)
sh(f"kubectl get pods -l {JOBSET}=big-no-gang,{RJOB}=workers")

Delete it so it stops holding the 2 free GPUs.

In [ ]:
sh("kubectl delete jobset big-no-gang --wait")

## ▶ NB 5.4 · Slide 29 (right): With per-role gangs: all or nothing

All 4 workers stay Pending and hold nothing. The driver's own gang (min 1) still runs.

In [ ]:
show("05-big.yaml")
sh("kubectl apply -f manifests/05-big.yaml")
wait_running("big", 1, rjob="driver")
time.sleep(10)
sh(f"kubectl get pods -l {JOBSET}=big,{RJOB}=workers")
sh(f"kubectl get podgroups -l {JOBSET}=big -o custom-columns="
   "NAME:.metadata.name,MIN:.spec.schedulingPolicy.gang.minCount,"
   "SCHEDULED:.status.conditions[0].status,REASON:.status.conditions[0].reason")
scheduling_tree("big")

## ▶ NB 5.5 · Slide 30: Shrink it in place, and the gang follows

`ElasticJobSet`: patch `parallelism`/`completions` on `workers` (index 1). The PodGroup's `minCount` is patched 4 → 2, and both remaining workers bind together.

In [ ]:
sh("""kubectl patch jobset big --type=json -p '[
  {"op":"replace","path":"/spec/replicatedJobs/1/template/spec/parallelism","value":1},
  {"op":"replace","path":"/spec/replicatedJobs/1/template/spec/completions","value":1}]'""")
wait_running("big", 2, rjob="workers")
sh(f"kubectl get podgroups -l {JOBSET}=big -o custom-columns=NAME:.metadata.name,MIN:.spec.schedulingPolicy.gang.minCount")
sh(f"kubectl get pods -l {JOBSET}=big,{RJOB}=workers")
scheduling_tree("big")

## ▶ NB 5.6 · Slide 31: Suspend releases the gang, resume brings it back

In [ ]:
sh("""kubectl patch jobset train --type=merge -p '{"spec":{"suspend":true}}'""")
wait_until("train's Workload and PodGroups deleted",
           lambda: not live(kjson(f"podgroups.scheduling.k8s.io -l {JOBSET}=train")["items"]))
sh(f"kubectl get workloads.scheduling.k8s.io,podgroups,pods -l {JOBSET}=train")

In [ ]:
sh("""kubectl patch jobset train --type=merge -p '{"spec":{"suspend":false}}'""")
wait_until("train's PodGroups recreated",
           lambda: len(live(kjson(f"podgroups.scheduling.k8s.io -l {JOBSET}=train")["items"])) == 2)
sh(f"kubectl get podgroups -l {JOBSET}=train")

## ▶ NB 6: Clean up

`reset` deletes every JobSet and Job but keeps the cluster for the next rehearsal. `down` deletes the cluster.

In [ ]:
sh("./cluster.sh reset")
# sh("./cluster.sh down")